In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import *

In [2]:
cnv="9:11857975-12047033_-"
name="9p23"

In [3]:
cmd=" ".join(["for f in $(find /oak/stanford/groups/mrivas/ukbb24983/cnv/gwas/",
              "-name '*.linear' -o -name '*.hybrid' | grep white_british);",
              "do",
              "phe=$(echo $f | awk -F'.' '{print $2}');"
              "awk -v p=$phe -v OFS='\\t' -v na='NA' -v g="+cnv,
              "'($NF < 0.01 && $3 == g && $NF !~ /na/){print p,$1,$2,$3,$4,$5,$6,$(NF-5),$(NF-4),$(NF-3),$(NF-2),$(NF-1),$NF}'",
              "$f;",
              "done",
              ">",
              "/oak/stanford/groups/jpriest/cnv_ukb/subanalyses/phewas/"+name+"_all_phewas_p001.tsv"
             ])
os.system(cmd)

0

In [4]:
phenos = pd.read_table("/oak/stanford/groups/jpriest/cnv_ukb/subanalyses/phewas/"+name+"_all_phewas_p001.tsv", 
                       usecols=[0,8,9,10,11,12],
                       names=["PHENO","N","BETA","SE","TSTAT","P"])
phenos.head()

,PHENO,N,BETA,SE,TSTAT,P


In [5]:
with open("/oak/stanford/groups/mrivas/users/magu/repos/rivas-lab/ukbb-tools/05_gbe/icdinfo.txt", "r") as i:
    names = {line.split()[0]:line.split()[2] for line in i}

phenos['BIN']  = phenos['PHENO'].apply(lambda x: 0 if 'QT' in x or 'INI' in x else 1)
phenos['BETA'] = phenos.apply(lambda row: np.log(row['BETA']) if row['BIN'] == 1 else row['BETA'], axis=1)
phenos['NAME'] = phenos['PHENO'].apply(lambda x: names.get(x) if x in names 
                                                 else names.get(x.replace('QT_FC', 'INI')) if x.replace('QT_FC','INI') in names 
                                                 else names.get(x.replace('BIN_FC','HC')))

phenos['-log10P'] = phenos['P'].apply(lambda x:-np.log10(x))

phenos.head(15)

ValueError: Cannot set a frame with no defined index and a value that cannot be converted to a Series

In [ ]:
phenos[(phenos['P'] < 0.001)].sort_values('P')

In [ ]:
# do the plot
filter_kw = lambda s: all([i not in s for i in ['3mm','method','_or_','doctor','generic','self-']])

y_ht = 3
fig  = plt.figure(figsize=(10, y_ht), dpi=300)
grid = plt.GridSpec(y_ht, 10, hspace=0.3)


for cnv_ix, cnv in enumerate([name]):
    # select significant data, truncate visualization
    data = phenos.loc[(~phenos['NAME'].isnull()) & 
                      (phenos['NAME'].apply(filter_kw)) & 
                      (phenos['PHENO'].apply(lambda s:'INI30' not in s))].query('P < 1e-3').copy()
    
    data = data.query('SE < 1').sort_values('P').head(50).copy()
    # group by QT/BIN status, plus fat measurement for second CNV
    order = [irow[0] for irow in sorted([irow for irow in data.iterrows()], 
                                         key=lambda (i,x): -x['P'] + 0*x['BIN'])] 

    # title
    fig.text(0.5, 0.9, cnv + ' PheWAS', horizontalalignment='center')
    
    # p-values
    ax = fig.add_subplot(grid[:,:4])
    ax.plot([data.loc[i,'-log10P'] for i in order], list(range(len(order))), 'ko')
    ax.set_ylim(-0.5,len(order)-0.5)
    ax.set_yticks(list(range(len(order)))) 
    ax.set_yticklabels(list(map(lambda s:s[0].upper() + s[1:].replace('_',' '), [data.loc[i,'NAME'] for i in order])))
    ax.set_xlabel('-log10(P)')
    ax.set_xlim(2,9)

    # betas
    ax = fig.add_subplot(grid[:,4:])
    ax.errorbar(x=[data.loc[i,'BETA'] for i in order], 
                y=list(range(len(order))),
                xerr=[data.loc[i,'SE'] for i in order],
                linestyle='', marker='D')
    ax.set_ylim(-0.5,len(order)-0.5)
    ax.set_yticks(list(range(len(order))))
    ax.set_yticklabels(['' for _ in order])
    ax.set_xlabel('LOR or BETA')
    ax.set_xlim(-1.5,5)
    ax.yaxis.set_label_position("right")

    # dash line for betas
    plt.plot([0,0], ax.get_ylim(), 'k--', linewidth=1)

plt.show()